## How to (not) break the Liber Primus like Rejewski

This notebook applies the method of `BreakingEnigma.ipynb` to a cipher nobody has broken: the
unsolved pages of Cicada 3301's *Liber Primus*. The arc is the same as the Enigma exercise:

1. a scenario and the materials,
2. a tutorial on the cipher machinery,
3. exploiting known structure (reproducing the pages that *are* solved),
4. statistics the key cannot hide,
5. a catalog of hypotheses, pruned by a statistic,
6. and then the ending is inverted. Rejewski's catalog collapsed 105,456 settings to a handful.
   Here the catalog collapses to nothing, and the last section shows, with a positive control,
   why that empty result is informative rather than a failure of the tools.

Everything is computed in the notebook from data files pinned by SHA-256. The cipher code and
data are vendored, unchanged except for file paths, from the validated `liber-primus` rig in
[https://github.com/Dukotah/cicada3301](https://github.com/Dukotah/cicada3301) (see `lp/__init__.py` and `lp/sources.py`). This notebook is a fork-only
extension of the NSA enigma-simulator exercise and is not affiliated with or endorsed by NSA.

### The scenario

In January 2014 the third Cicada 3301 puzzle led to a 58-page book written in runes, the
*Liber Primus*. A handful of its pages were decrypted within weeks, with ciphers no stronger
than a Vigenere square. The rest, about thirteen thousand runes, have resisted a decade of
public effort. You have the transcription, the solved pages, and a laptop.

### Materials

- `lp/data/krisyotam_runes.txt`: the transcription, pages separated by `%`. Its unsolved part is
  identified by the SHA-256 of its comma-joined rune indices, published in the rig's
  `PROBLEM.json`; if your copy does not hash identically, no number here is comparable to yours.
- `lp/data/scream314_lp.md`: a page-by-page write-up including the solved pages.
- `lp/data/SOLVED-PAGES.json`: the plaintexts of the solved pages as derived by the rig.
- `lp/data/english_quadgrams.txt`: English quadgram counts, used as a fitness score.
- `lp/data/keys/`: candidate key material.

In [1]:
from lp import sources, gematria as gp, ciphers, stats, solve, corpus, catalog
from lp import skipdecode as sk
import hashlib, math, random

for name, ok in sources.verify().items():
    print('%-28s %s' % (name, 'sha256 OK' if ok else 'SHA256 MISMATCH'))
assert all(sources.verify().values())

segs = sources.unsolved_segments()
U = sources.unsolved_stream()
pinned = sources.problem()['ciphertext_identity']['sha256_of_comma_joined_indices']
print()
print('unsolved segments:', len(segs), ' runes:', len(U))
print('sha256 of the stream:', sources.stream_sha256(U))
print('matches PROBLEM.json:', sources.stream_sha256(U) == pinned)

english_quadgrams.txt        sha256 OK
krisyotam_runes.txt          sha256 OK
scream314_lp.md              sha256 OK
PROBLEM.json                 sha256 OK
SOLVED-PAGES.json            sha256 OK
keys/self_reliance.txt       sha256 OK
keys/solved_plaintext.txt    sha256 OK
keys/thematic.txt            sha256 OK



unsolved segments: 55  runes: 12956
sha256 of the stream: 023312066df471005264b9cbe7997cb77d2a9a6a2dc9b3316d22674023af1585
matches PROBLEM.json: True


### The machinery: Gematria Primus

The book uses 29 runes. Each rune has a position (0 to 28, the value used for shifting), a
transliteration (seven runes stand for two letters: TH, EO, NG, OE, AE, IA, EA), and a prime
number. All key arithmetic is modulo 29.

In [2]:
for i, rune, tr, prime in gp.GEMATRIA:
    print('%2d  %s  %-2s  %3d' % (i, rune, tr, prime))

 0  ᚠ  F     2
 1  ᚢ  U     3
 2  ᚦ  TH    5
 3  ᚩ  O     7
 4  ᚱ  R    11
 5  ᚳ  C    13
 6  ᚷ  G    17
 7  ᚹ  W    19
 8  ᚻ  H    23
 9  ᚾ  N    29
10  ᛁ  I    31
11  ᛄ  J    37
12  ᛇ  EO   41
13  ᛈ  P    43
14  ᛉ  X    47
15  ᛋ  S    53
16  ᛏ  T    59
17  ᛒ  B    61
18  ᛖ  E    67
19  ᛗ  M    71
20  ᛚ  L    73
21  ᛝ  NG   79
22  ᛟ  OE   83
23  ᛞ  D    89
24  ᚪ  A    97
25  ᚫ  AE  101
26  ᚣ  Y   103
27  ᛡ  IA  107
28  ᛠ  EA  109


In [3]:
word = 'DIVINITY'
idx = gp.keyword_to_indices(word)
print(word, '->', idx)
print('runes   ', gp.indices_to_runes(idx))
print('primes  ', [gp.PRIMES[i] for i in idx], ' sum =', sum(gp.PRIMES[i] for i in idx))
print('back    ', gp.indices_to_translit(idx, '-'))

DIVINITY -> [23, 10, 1, 10, 9, 10, 16, 26]
runes    ᛞᛁᚢᛁᚾᛁᛏᚣ
primes   [89, 31, 3, 31, 29, 31, 59, 103]  sum = 376
back     D-I-U-I-N-I-T-Y


Every cipher here is a keystream `k` applied position by position: encryption `c = p + k` and
decryption `p = c - k` (mod 29), or the reverse sign. Atbash reflects the alphabet, `i -> 28 - i`.
A Caesar shift is a constant keystream, Vigenere repeats a keyword, a running key reads a book,
and the solved pages also use a keystream made from the primes.

**Exercise 1.** Write `shift(idxs, k)` and `atbash(idxs)` over rune indices.

In [4]:
def shift(idxs, k):
    return [(c + k) % 29 for c in idxs]

def atbash(idxs):
    return [28 - c for c in idxs]

Round trips for each cipher family, using the rig's implementations:

In [5]:
msg = gp.keyword_to_indices('THE PRIMES ARE SACRED')
keystreams = {
    'shift 7':            [7] * len(msg),
    'vigenere DIVINITY':  ciphers.repeat_key(gp.keyword_to_indices('DIVINITY'), len(msg)),
    'running key':        sk.eng_to_idx(sources.keytext('self_reliance.txt'))[:len(msg)],
    'primes':             ciphers.prime_stream(len(msg)),
    'totient of primes':  ciphers.prime_totient_stream(len(msg)),
}
for name, K in keystreams.items():
    c = ciphers.apply_stream_to_indices(msg, K, sign=+1)
    p = ciphers.apply_stream_to_indices(c, K, sign=-1)
    assert p == msg
    print('%-18s %s  ->  %s' % (name, gp.indices_to_runes(c), gp.indices_to_translit(p)))
c = ciphers.atbash_indices(msg)
assert ciphers.atbash_indices(c) == msg
print('%-18s %s  ->  %s' % ('atbash', gp.indices_to_runes(c), gp.indices_to_translit(ciphers.atbash_indices(c))))

shift 7            ᚾᚫᛚᛄᛒᚣᚫᛟᚦᛄᚫᛟᚦᛇᛄᚫᚢ  ->  THEPRIMESARESACRED
vigenere DIVINITY  ᚫᛠᛉᛉᛗᚠᚳᛇᛖᛉᛗᚫᚱᛋᛚᛋᛒ  ->  THEPRIMESARESACRED
running key        ᛒᚹᚱᚱᛉᚻᚾᛈᚱᚾᚹᚪᛈᛝᛟᛞᚪ  ->  THEPRIMESARESACRED
primes             ᚱᛝᛖᛄᛝᚩᚷᚳᛖᚱᛚᛞᚹᛗᛟᛈᚪ  ->  THEPRIMESARESACRED
totient of primes  ᚩᛚᛒᛁᛚᚦᚳᚱᛒᚩᛗᛟᚷᛖᛝᛇᛞ  ->  THEPRIMESARESACRED
atbash             ᚣᛁᛋᚪᛖᚾᛁᛈᚱᚪᛁᛈᚱᛞᚪᛁᚳ  ->  THEPRIMESARESACRED


### Exploiting known structure: the solved pages

Rejewski had the doubled message key. The Liber Primus solver has the solved pages: they show
which transforms the author used, and they are the trust anchor for every tool below. A tool
that cannot re-derive them is not worth running on the unsolved pages (the rig enforces this in
`tests/validate.py`, which this section follows).

The fitness score is the mean log10 probability of the transliteration's quadgrams under an
English model: real English sits far above random letters.

In [6]:
for s in ['WELCOMEPILGRIMTOTHEGREATJOURNEY', 'QXZJCUWPQXZJCUWPQXZJCUWP']:
    print('%7.3f  %s' % (catalog.fitness(gp.keyword_to_indices(s)), s))

 -4.405  WELCOMEPILGRIMTOTHEGREATJOURNEY
 -8.861  QXZJCUWPQXZJCUWPQXZJCUWP


**Exercise 2.** Three pages need nothing more than Atbash and a shift. Write a brute force over
{direct, Atbash} x {29 shifts} x {both signs}, rank the candidates by `catalog.fitness`, and
apply it to the page labelled `Runes - 01.jpg` (`corpus.page_by_label(label)['runes']`, then
`gp.runes_to_indices`).

In [7]:
def brute_simple(idxs):
    cands = []
    for atb in (False, True):
        base = atbash(idxs) if atb else list(idxs)
        for k in range(29):
            for sign in (-1, 1):
                out = shift(base, sign * k)
                label = ('atbash+' if atb else '') + 'shift%+d' % (sign * k)
                cands.append((catalog.fitness(out), label, out))
    return sorted(cands, key=lambda x: x[0], reverse=True)

sc, label, out = brute_simple(gp.runes_to_indices(corpus.page_by_label('Runes - 01.jpg')['runes']))[0]
print(label, round(sc, 3), gp.indices_to_translit(out)[:80])

atbash+shift+0 -4.478 AWARNNGBELIEUENOTHNGFROMTHISBOOCEXCEPTWHATYOUCNOWTOBETRUETESTTHECNOWLEDGEFINDYOU


The reference brute force, applied to all three pages, checked against `SOLVED-PAGES.json`:

In [8]:
from lp.reference import brute_simple as ref_brute
solved = {p['page_label']: p for p in sources.solved_pages()['pages']}

def page_runes(label):
    return corpus.page_by_label(label)['runes']

def check(label, text):
    expected = solved[label]['plaintext_transliteration']
    print('  matches SOLVED-PAGES.json:', text == expected, '| %s' % solved[label]['title'])

for label in ['Runes - 01.jpg', '05.jpg', '06.jpg']:
    sc, how, out = ref_brute(gp.runes_to_indices(page_runes(label)))[0]
    text = gp.indices_to_translit(out)
    print('%-15s %-14s fitness %.3f  %s' % (label, how, sc, text[:70]))
    check(label, text)

Runes - 01.jpg  atbash+shift+0 fitness -4.478  AWARNNGBELIEUENOTHNGFROMTHISBOOCEXCEPTWHATYOUCNOWTOBETRUETESTTHECNOWLE
  matches SOLVED-PAGES.json: True | A WARNING


05.jpg          shift+0        fitness -4.979  SOMEWISDOMTHEPRIMESARESACREDTHETOTIENTFUNCTIANISSACREDALLTHNGSSHOULDBE
  matches SOLVED-PAGES.json: True | SOME WISDOM


06.jpg          atbash+shift+3 fitness -4.127  ACOANAMANDECIDEDTOGOANDSTUDYWITHAMASTERHEWENTTOTHEDOOROFTHEMASTERWHOAR
  matches SOLVED-PAGES.json: True | A KOAN (the master and the student)


The next three pages use a real keystream, and they introduce the *interrupter*: on solved
pages some occurrences of the rune F (index 0) are not enciphered and do not advance the key.
Which ones is not marked, so `solve.find_interrupters` runs a beam search over the keep/skip
choice for every F, scored by the same quadgram model.

**Exercise 3.** Before running the beam, decrypt page `03.jpg` with the key `DIVINITY` rigidly
(`ciphers.repeat_key`, `ciphers.apply_stream_to_indices` with `sign=-1`) and find where the
plaintext stops making sense. Which rune sits there? (Not every F is an interrupter.)

In [9]:
c = gp.runes_to_indices(page_runes('03.jpg'))
p = ciphers.apply_stream_to_indices(c, ciphers.repeat_key(gp.keyword_to_indices('DIVINITY'), len(c)), sign=-1)
print(gp.indices_to_translit(p)[:120])
f_positions = [i for i, x in enumerate(c) if x == 0]
print('F runes at positions', f_positions)
# Where does it break? Compare with the interrupter search used in the next cell.
res = solve.find_interrupters(page_runes('03.jpg'), ciphers.repeat_key(gp.keyword_to_indices('DIVINITY'), len(c)), sign=-1, beam_width=500)
first = f_positions[res['interrupters'][0]]
print('first interrupter is F number %d, at position %d' % (res['interrupters'][0], first))
print('rigid plaintext up to it:', gp.indices_to_translit(p[:first]))
print('and after it:            ', gp.indices_to_translit(p[first:first + 20]))

WELCOMEWELCOMEPILGRIMTOTHEGREATJOURNEYTOWARDTHEENDOGHFJOLNFMDAFRSEEATHEAGWCNWWRFEAOAETHAEEAJDUOLUOGEAGBHSLLCIXDBUNWAEAUT
F runes at positions [5, 14, 47, 48, 74, 84, 132, 144, 152, 159, 160, 165, 219, 250, 317, 331]


first interrupter is F number 2, at position 47
rigid plaintext up to it: WELCOMEWELCOMEPILGRIMTOTHEGREATJOURNEYTOWARDTHEEND
and after it:             OGHFJOLNFMDAFRSEEATHEAG


In [10]:
jobs = [
    ('03.jpg', 'Vigenere DIVINITY',         lambda n: ciphers.repeat_key(gp.keyword_to_indices('DIVINITY'), n)),
    ('14.jpg', 'Vigenere FIRFUMFERENFE',    lambda n: ciphers.repeat_key(gp.keyword_to_indices('FIRFUMFERENFE'), n)),
    ('73.jpg', 'totient of the primes, p-1', lambda n: ciphers.prime_totient_stream(n)),
]
for label, how, make in jobs:
    runes = page_runes(label)
    res = solve.find_interrupters(runes, make(len(gp.runes_to_indices(runes))), sign=-1, beam_width=500)
    print('%-7s %-27s %d interrupters, fitness %.3f' % (label, how, res['n_interrupters'], res['score_norm']))
    print('  ', res['plaintext'][:90])
    expected = solved.get(label, {}).get('plaintext_transliteration')
    if expected:
        print('   matches SOLVED-PAGES.json:', res['plaintext'] == expected, '|', solved[label]['title'])

03.jpg  Vigenere DIVINITY           7 interrupters, fitness -4.344
   WELCOMEWELCOMEPILGRIMTOTHEGREATJOURNEYTOWARDTHEENDOALLTHNGSITISNOTANEASYTRIPBUTORTHOSEWHOI
   matches SOLVED-PAGES.json: True | WELCOME


14.jpg  Vigenere FIRFUMFERENFE      2 interrupters, fitness -4.243
   ACOANDURNGALESSONTHEMASTEREXPLAINEDTHEITHEIISTHEUOICEOTHECIRCUMERENCEHESAIDWHENASCEDBYASTU
   matches SOLVED-PAGES.json: True | A KOAN (the circumference)


73.jpg  totient of the primes, p-1  1 interrupters, fitness -4.573
   ANENDWITHINTHEDEEPWEBTHEREEXISTSAPAGETHATHASHESTOITISTHEDUTYOEUERYPILGRIMTOSEECOUTTHISPAGE


Page `73.jpg` is AN END: its key is `prime - 1` (Euler's totient of a prime) over consecutive
primes, exactly the rule stated on page 05, "the primes are sacred, the totient function is
sacred". Every transform the author is known to have used is now reproduced.

### What the key cannot hide

The Enigma cycle structure survived any plugboard. The equivalent here is a set of statistics a
cipher of this kind cannot disguise:

- **index of coincidence** times 29: about 1.0 for uniform random runes, higher for language;
- **flatness** of the rune frequencies (chi-square against uniform, Shannon entropy);
- **doublet rate**: how often a rune is followed by the same rune. For uniform random runes it
  is 1/29.

**Exercise 4.** Write `doublet_rate(idxs)` and compute it for the unsolved stream `U`.

In [11]:
def doublet_rate(idxs):
    return sum(a == b for a, b in zip(idxs, idxs[1:])) / (len(idxs) - 1)

print('%.4f %%' % (100 * doublet_rate(U)))

0.6638 %


In [12]:
def row(name, idxs):
    print('%-34s %6d  %6.3f  %8.1f  %6.3f  %6.2f %%' % (name, len(idxs), stats.ioc_norm(idxs),
          stats.chi2_uniform(idxs), stats.shannon_entropy(idxs), 100 * stats.doublet_rate(idxs)))

print('%-34s %6s  %6s  %8s  %6s  %8s' % ('', 'runes', 'IoC*29', 'chi2', 'H bits', 'doublets'))
for label in ['Runes - 01.jpg', '06.jpg', '03.jpg', '14.jpg']:
    row('solved ciphertext ' + label, gp.runes_to_indices(page_runes(label)))
english = sk.eng_to_idx(sources.keytext('self_reliance.txt'))
row('English (Self-Reliance)', english)
rng = random.Random(29)
row('uniform random', [rng.randrange(29) for _ in range(len(U))])
row('UNSOLVED stream', U)
print('log2(29) = %.4f bits, 1/29 = %.2f %%' % (math.log2(29), 100 / 29))

                                    runes  IoC*29      chi2  H bits  doublets


solved ciphertext Runes - 01.jpg      184   1.857     184.8   4.165    2.73 %


solved ciphertext 06.jpg              742   1.948     730.4   4.113    1.89 %


solved ciphertext 03.jpg              394   1.184     100.2   4.643    3.56 %


solved ciphertext 14.jpg              319   1.130      69.5   4.706    2.83 %


English (Self-Reliance)             41875   1.804   33679.3   4.239    2.84 %
uniform random                      12956   0.999      18.0   4.857    3.59 %
UNSOLVED stream                     12956   1.000      26.4   4.857    0.66 %
log2(29) = 4.8580 bits, 1/29 = 3.45 %


The unsolved stream is as flat as uniform noise on every frequency measure, so no Caesar,
Atbash or short Vigenere of English is possible (those leave the language's peaks visible, as
the solved ciphertexts do). But its doublet rate is far *below* random. How far:

In [13]:
n_pairs = len(U) - 1
observed = stats.doublet_count(U)
expected = n_pairs / 29
z = (observed - expected) / math.sqrt(n_pairs * (1 / 29) * (28 / 29))
print('doublets observed %d, expected for random %.1f, z = %.2f' % (observed, expected, z))
low = [i for i, s in enumerate(segs) if stats.doublet_rate(s) < 1 / 29]
print('segments below the random rate: %d of %d' % (len(low), len(segs)))

doublets observed 86, expected for random 446.7, z = -17.37
segments below the random rate: 55 of 55


Could an ordinary keyed cipher produce that? Encipher English with each family and measure
(the plaintext is Self-Reliance, the same length as the unsolved stream):

In [14]:
P = english[:len(U)]
running = sk.eng_to_idx(sources.keytext('solved_plaintext.txt'))
running = (running * (len(P) // len(running) + 1))[:len(P)]
rng = random.Random(3301)
families = {
    'Vigenere DIVINITY':          ciphers.repeat_key(gp.keyword_to_indices('DIVINITY'), len(P)),
    'totient-of-primes stream':   ciphers.prime_totient_stream(len(P)),
    'running key (English book)': running,
    'one-time pad (uniform)':     [rng.randrange(29) for _ in P],
}
for name, K in families.items():
    C = ciphers.apply_stream_to_indices(P, K, sign=+1)
    print('%-28s doublets %5.2f %%   IoC*29 %.3f' % (name, 100 * stats.doublet_rate(C), stats.ioc_norm(C)))
print('%-28s doublets %5.2f %%   IoC*29 %.3f' % ('LP2 unsolved', 100 * stats.doublet_rate(U), stats.ioc_norm(U)))

Vigenere DIVINITY            doublets  3.64 %   IoC*29 1.188
totient-of-primes stream     doublets  3.05 %   IoC*29 1.002
running key (English book)   doublets  3.24 %   IoC*29 1.029
one-time pad (uniform)       doublets  3.61 %   IoC*29 1.000
LP2 unsolved                 doublets  0.66 %   IoC*29 1.000


Every additive cipher leaves doublets near or above the random rate, whatever the key. The
deficit is therefore not something a better key explains; something in the enciphering process
refuses to write the same rune twice. Hold that thought until the last section.

### The hypothesis catalog

Rejewski's catalog listed, for every rotor setting, the cycle structure it produces. The
analogue is a library of candidate keystreams, each tried against every unsolved segment, and
scored. `lp/catalog.py` builds one from what the solved pages suggest:

- the prime, totient-of-prime and totient(n) streams, started at offsets 0 to 49;
- Vigenere with each thematic keyword in `keys/thematic.txt`, at every key phase;
- running keys from the solved-page plaintexts and from Emerson's *Self-Reliance*, every 100 runes;

each in both directions, with and without Atbash, against the first 120 runes of each of the
unsolved segments. The same catalog is also run on order-shuffled copies of the segments: the
shuffle keeps the rune frequencies and destroys any real decryption, so it is the null.

The sweep takes about a minute; `make_lp_cache.py` stores it in `lp/cache/catalog.json`, and
`catalog.load_or_run()` recomputes it if the cache is missing or stale.

In [15]:
lib = catalog.library()
fams = {}
for name, _ in lib:
    fams[name.split('@')[0].split(':')[0]] = fams.get(name.split('@')[0].split(':')[0], 0) + 1
print('candidate keystreams:', len(lib), fams)

res = catalog.load_or_run()
real, null = res['real'], res['null']
print('catalog result from:', res['source'])
print('decodes scored: %d real, %d null' % (real['n'], null['n']))
print()
print('best real decodes:')
for sc, target, key, sign, atb in real['top'][:5]:
    print('  %.3f  %-11s %-34s sign %+d%s' % (sc, target, key, sign, ' atbash' if atb else ''))

candidate keystreams: 991 {'primes': 50, 'prime-1': 50, 'totient(n)': 50, 'vigenere': 333, 'running': 508}


catalog result from: cache
decodes scored: 218020 real, 218020 null

best real decodes:
  -6.330  segment 49  vigenere:WITHIN/0                  sign -1 atbash
  -6.392  segment 53  vigenere:ENCRYPTED/3               sign +1
  -6.446  segment 50  vigenere:ENCRYPT/4                 sign -1 atbash
  -6.472  segment 2   vigenere:TOTIENT/0                 sign +1
  -6.477  segment 54  primes@30                          sign +1 atbash


Spot-check the cache by recomputing its best real decode live:

In [16]:
sc, target, key, sign, atb = real['top'][0]
C = segs[int(target.split()[-1])][:catalog.L]
if atb:
    C = ciphers.atbash_indices(C)
K = dict(lib)[key]
p = [(a + sign * b) % 29 for a, b in zip(C, K)]
print('recomputed %.6f, cached %.6f' % (catalog.fitness(p), sc))
assert abs(catalog.fitness(p) - sc) < 1e-9
print(gp.indices_to_translit(p))

recomputed -6.329688, cached -6.329688
DFSEEAEAHNGHIASTWODSDIAEANRLINIATHTINGDIFEEADEOTHEANIARIBUMNHDGXMGUEBLRDOEEAGMWGIA


Is that best decode a lead? The maximum of N draws from a null grows with N. For a roughly
Gaussian null with standard deviation sd, the expected maximum of N draws sits near
`mean + sd * sqrt(2 ln N)`; a single "best of N" result has to clear at least that before it
means anything. (The rig's `benchmark/null.py` goes further and calibrates the bar from observed
extreme values, because the bulk standard deviation can misjudge the tail. Comparing with the
maximum of a same-size null, as below, is the direct check.)

**Exercise 5.** From `null['mean']`, `null['sd']` and `real['n']`, compute that bar.

In [17]:
bar = null['mean'] + null['sd'] * math.sqrt(2 * math.log(real['n']))
print('%.3f' % bar)

-6.378


In [18]:
gauss_bar = null['mean'] + null['sd'] * math.sqrt(2 * math.log(real['n']))
print('null:  mean %.3f  sd %.3f' % (null['mean'], null['sd']))
print('mean + sd*sqrt(2 ln N), N = %d:  %.3f' % (real['n'], gauss_bar))
print('max of the same-size shuffled null:     %.3f' % null['max'])
print('max of the real catalog:                %.3f' % real['max'])
print()
print('real max above the shuffled-null max?', real['max'] > null['max'])

null:  mean -7.505  sd 0.227
mean + sd*sqrt(2 ln N), N = 218020:  -6.378
max of the same-size shuffled null:     -6.207
max of the real catalog:                -6.330

real max above the shuffled-null max? False


Two things to notice. The shuffled null's own maximum lands above the Gaussian bar: the tail of
this score distribution is heavier than a Gaussian with the same spread, so the formula
understates where the best wrong answer falls. That is why the rig calibrates its bars from
observed extremes, and why the honest comparison is with a same-size null. Judged against the
Gaussian bar alone, the best real decode would look like a lead; it is gibberish, and it does
not beat the null.

Positive controls through the very same catalog: the solved keystream pages.

In [19]:
for label in ['73.jpg', '03.jpg', '14.jpg']:
    out = catalog.sweep([(label, gp.runes_to_indices(page_runes(label)))], lib)
    sc, _, key, sign, atb = out['top'][0]
    print('%-7s best %-28s fitness %.3f  above the real-catalog max: %s, above the null max: %s'
          % (label, key, sc, sc > real['max'], sc > null['max']))

73.jpg  best prime-1@0                    fitness -5.282  above the real-catalog max: True, above the null max: True


03.jpg  best vigenere:DIVINITY/5          fitness -6.104  above the real-catalog max: True, above the null max: True


14.jpg  best vigenere:FIRFUMFERENFE/11    fitness -6.197  above the real-catalog max: True, above the null max: True


AN END is found by the catalog, at the top, with its true key. The two Vigenere pages are found
with their true keywords too, but only marginally: the catalog decodes rigidly, and the first
interrupter knocks the key out of step for the rest of the window. That is the first hint of
the lesson below: an instrument only finds what its decoding model can represent.

Meanwhile the real unsolved segments produce nothing the shuffled null does not also produce.
The catalog lookup came back empty.

### The inverted ending: why the empty result is informative

Rejewski could trust his catalog because he could check it on a day key he already knew. Do
the same here: **plant** a known key, and ask whether the instrument recovers it. A negative
from an instrument that has never recovered a planted signal is not a negative. (This is the
rig's central rule; see `benchmark/README.md`.)

**Plant A**: a genuine Liber Primus plaintext (A WARNING) enciphered with a running key that is
in the catalog (the solved-page text at offset 3000), with no filter.

In [20]:
warning = sk.eng_to_idx(solved['Runes - 01.jpg']['plaintext_transliteration'])
keybook = sk.eng_to_idx(sources.keytext('solved_plaintext.txt'))
plant_offset = 3000
CA = [(p - (-1) * k) % 29 for p, k in zip(warning, keybook[plant_offset:])]   # decode is p = c - k
out = catalog.sweep([('plant A', CA)], lib)
sc, _, key, sign, atb = out['top'][0]
print('best: %s sign %+d  fitness %.3f   (planted: running:solved_plaintext.txt@%d)' % (key, sign, sc, plant_offset))
print('clears the real-catalog max: %s; doublet rate of plant A: %.2f %%' % (sc > real['max'], 100 * stats.doublet_rate(CA)))

best: running:solved_plaintext.txt@3000 sign -1  fitness -4.763   (planted: running:solved_plaintext.txt@3000)
clears the real-catalog max: True; doublet rate of plant A: 3.83 %


Found at once. **Plant B** is the same plaintext and key, enciphered through the
*anti-repeat filter* the rig infers for the Liber Primus: whenever the next ciphertext rune
would repeat the previous one, the encipherer usually rejects that key symbol and moves to the
next (`skipdecode.encipher_keyskip`, at the rig's pinned rejection probability `0.83`).

**Exercise 6.** Before running the next cell, predict: will the catalog find plant B?

In [21]:
CB, skips, _ = sk.encipher_keyskip(warning, keybook[plant_offset:], sign=-1, supp=0.83, seed=3301)
print('key symbols skipped:', sum(skips), ' doublet rate of plant B: %.2f %%' % (100 * stats.doublet_rate(CB)))
out = catalog.sweep([('plant B', CB)], lib)
sc, _, key, sign, atb = out['top'][0]
print('catalog best: %s  fitness %.3f   clears the shuffled-null max: %s' % (key, sc, sc > null['max']))

rigid = sk.rigid_decode(CB, keybook, sign=-1, o=plant_offset)
beam = sk.beam_decode(CB, keybook, sign=-1, o=plant_offset, beam_w=500, max_skip=3)
wrong = sk.beam_decode(CB, english, sign=-1, o=1234, beam_w=500, max_skip=3)
def recovered(d):
    return sum(a == b for a, b in zip(d['plain_idx'], warning)) / len(warning)
print()
print('correct key, rigid decode     fitness %.3f  runes recovered %5.1f %%' % (rigid['score'], 100 * recovered(rigid)))
print('correct key, skip-aware beam  fitness %.3f  runes recovered %5.1f %%' % (beam['score'], 100 * recovered(beam)))
print('wrong key,   skip-aware beam  fitness %.3f  runes recovered %5.1f %%' % (wrong['score'], 100 * recovered(wrong)))
print()
print(beam['translit'][:90])

key symbols skipped: 6  doublet rate of plant B: 0.55 %


catalog best: totient(n)@32  fitness -6.635   clears the shuffled-null max: False



correct key, rigid decode     fitness -7.524  runes recovered  14.1 %
correct key, skip-aware beam  fitness -4.478  runes recovered 100.0 %
wrong key,   skip-aware beam  fitness -7.169  runes recovered   7.6 %

AWARNNGBELIEUENOTHNGFROMTHISBOOCEXCEPTWHATYOUCNOWTOBETRUETESTTHECNOWLEDGEFINDYOURTRUTHEXPE


With the filter, even the *correct* key decoded rigidly scores as noise: each rejected key
symbol shifts the alignment for everything after it. A decoder that tracks the skips recovers
the text, and a wrong key stays noise under the same decoder. So the catalog above was blind to
exactly this kind of cipher, and the rig's sweeps use the skip-aware decoder for that reason.

Now the decisive question: what does the filter do to the statistics? Encipher English with a
true one-time pad through the filter, and separately with a keystream *derived from a short
seed* (SHA-256 in counter mode, reduced to 0..28 by rejection sampling), and compare both with
the unsolved stream.

In [22]:
def sha256_ctr(seed, n):
    out, ctr = [], 0
    while len(out) < n:
        out.extend(b % 29 for b in hashlib.sha256(seed + ctr.to_bytes(4, 'big')).digest() if b < 232)
        ctr += 1
    return out[:n]

rng = random.Random(1)
pad = [rng.randrange(29) for _ in range(2 * len(P))]
derived = sha256_ctr(b'CICADA3301', 2 * len(P))

def signature(name, C):
    print('%-36s doublets %5.2f %%   IoC*29 %.4f   H %.4f bits' % (name, 100 * stats.doublet_rate(C),
          stats.ioc_norm(C), stats.shannon_entropy(C)))

signature('one-time pad + filter', sk.encipher_keyskip(P, pad, sign=-1, supp=0.83, seed=7)[0])
signature('SHA-256(short seed) + filter', sk.encipher_keyskip(P, derived, sign=-1, supp=0.83, seed=7)[0])
signature('running key (English) + filter', sk.encipher_keyskip(P, running + running, sign=-1, supp=0.83, seed=7)[0])
signature('LP2 unsolved', U)

one-time pad + filter                doublets  0.61 %   IoC*29 1.0001   H 4.8563 bits
SHA-256(short seed) + filter         doublets  0.58 %   IoC*29 0.9994   H 4.8569 bits
running key (English) + filter       doublets  0.55 %   IoC*29 1.0328   H 4.8318 bits
LP2 unsolved                         doublets  0.66 %   IoC*29 0.9999   H 4.8565 bits


Read the rows against the bottom one. A pad or a derived keystream passed through the filter
lands in the same regime as the unsolved stream: flat frequencies, entropy at the maximum, and
the doublet deficit. The running-key row does not: with language in both plaintext and key, its
index of coincidence stays visibly above 1 and its entropy below the maximum, which the
unsolved stream does not show. Nothing in the ciphertext statistics separates a true pad from a keystream
expanded from a short secret. That is what "OTP-class" means in the Cicada repository, and it
is all it means. The rig's own statement of the verdict, read from `PROBLEM.json`:

In [23]:
verdict = sources.problem()['current_verdict']
print(verdict['statement'])
print()
print('DO NOT SAY:', verdict['do_not_say'])

OTP-class. The ciphertext is indistinguishable between a TRUE EXTERNAL PAD (information-theoretically closed) and a keystream DERIVED FROM A SHORT SEED (finite keyspace, brute-forceable). Only running the derived-key search settles which.

DO NOT SAY: Do NOT state 'information-theoretically unsolvable' without that qualification. That was this repo's own overreach, caught by round12/D3, and it foreclosed a real tractable lane.


So the inverted ending is precise:

- the unsolved pages behave like a one-time pad through an anti-repeat filter, and every
  frequency statistic is flat;
- the filter explains the doublet deficit that no ordinary keyed cipher can produce;
- an instrument that decodes rigidly cannot find even the correct key under that filter, while
  the skip-aware decoder can (the positive control above);
- and none of this proves the pages unsolvable. A keystream derived from a short seed is not
  excluded, which is why the Cicada repository still runs seed sweeps.

Where to go from here:

- the full rig, its validation gate and benchmark: [https://github.com/Dukotah/cicada3301](https://github.com/Dukotah/cicada3301) (`liber-primus/`)
- the methods paper, *Every Negative Was an English-Only Negative*: [`PAPER.md`](https://github.com/Dukotah/cicada3301/blob/master/PAPER.md)
- lanes that are still open: [`docs/OPEN-LANES.md`](https://github.com/Dukotah/cicada3301/blob/master/docs/OPEN-LANES.md)
- donate compute to a parked seed sweep: [`liber-primus/volunteer/`](https://github.com/Dukotah/cicada3301/blob/master/liber-primus/volunteer)
- if you think you have a solution, the oracle and its acceptance criteria: `PROBLEM.json`,
  `verify_solution.py` in [`liber-primus/`](https://github.com/Dukotah/cicada3301/blob/master/liber-primus)

Sources: the Gematria Primus table and solved-page transforms follow the community record
compiled by scream314 ([github.com/scream314/cicada3301](https://github.com/scream314/cicada3301));
the transcription is the rtkd/iddqd 2017 lineage; Rejewski's method is described in
`BreakingEnigma.ipynb` and in M. Rejewski, "How Polish Mathematicians Deciphered the Enigma",
*Annals of the History of Computing* 3(3), 1981.